# LGBM 모델링 과정



In [175]:
# 데이터 로드-> train,test 스플릿-> EDA-> 결측치 제거->이상치 조정->피처 생성->인코딩/스케일링->모델링->CV->모델평가->모델 저장

In [176]:
from google.colab import drive
drive.mount('/content/data')

Drive already mounted at /content/data; to attempt to forcibly remount, call drive.mount("/content/data", force_remount=True).


In [177]:
! pip install optuna

# 데이터 로드 및 확인

In [178]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import lightgbm as lgb
import optuna


In [179]:
import os
import random
import torch
# 랜덤 시드 고정 함수
def reset_seeds(seed=42):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)    # 파이썬 환경변수 시드 고정
  np.random.seed(seed)
  torch.manual_seed(seed) # cpu 연산 무작위 고정
  torch.cuda.manual_seed(seed) # gpu 연산 무작위 고정
  torch.backends.cudnn.deterministic = True  # cuda 라이브러리에서 Deterministic(결정론적)으로 예측하기 (예측에 대한 불확실성 제거 )


In [180]:
reset_seeds()
import easydict
args= easydict.EasyDict()


# 원본데이터 path
args.data_csv='/content/data/MyDrive/2차 프로젝트/data/dataset.csv'

# 모델링 파일을 저장
args.submission_pkl= '/content/data/MyDrive/2차 프로젝트/LGBM_01.pkl'

In [181]:
dataset=pd.read_csv(args.data_csv)

In [182]:
dataset.shape

(5630, 20)

In [183]:
dataset.columns

Index(['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier',
       'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp',
       'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore',
       'MaritalStatus', 'NumberOfAddress', 'Complain',
       'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
       'DaySinceLastOrder', 'CashbackAmount'],
      dtype='object')

In [184]:
dataset.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130


## 데이터 타입 변경

In [185]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

In [186]:
df_object = dataset.select_dtypes(include='object')
df_object.columns
df_int = dataset.select_dtypes(include='int64')
df_int.columns
df_float = dataset.select_dtypes(include='float64')
df_float.columns

for col in df_object.columns:
  dataset[col]=dataset[col].astype('category')

for col in df_int.columns:
  dataset[col]=dataset[col].astype('int32')

for col in df_float.columns:
  dataset[col]=dataset[col].astype('float32')

In [187]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   CustomerID                   5630 non-null   int32   
 1   Churn                        5630 non-null   int32   
 2   Tenure                       5366 non-null   float32 
 3   PreferredLoginDevice         5630 non-null   category
 4   CityTier                     5630 non-null   int32   
 5   WarehouseToHome              5379 non-null   float32 
 6   PreferredPaymentMode         5630 non-null   category
 7   Gender                       5630 non-null   category
 8   HourSpendOnApp               5375 non-null   float32 
 9   NumberOfDeviceRegistered     5630 non-null   int32   
 10  PreferedOrderCat             5630 non-null   category
 11  SatisfactionScore            5630 non-null   int32   
 12  MaritalStatus                5630 non-null   category
 13  Num

# train/test 스플릿

In [188]:
from sklearn.model_selection import train_test_split

In [189]:
reset_seeds()
y = dataset['Churn'] # target data
X = dataset.drop(['Churn'], axis=1) #feature data

In [190]:
reset_seeds()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=dataset['Churn'])

In [191]:
X_tr.shape, X_te.shape, y_tr.shape, y_te.shape

((3941, 19), (1689, 19), (3941,), (1689,))

In [192]:
X_tr.shape[0] == y_tr.shape[0] , X_te.shape[0] == y_te.shape[0]

(True, True)

In [193]:
X_tr.shape[1] == X_te.shape[1]

True

In [194]:
X_tr['CustomerID'].nunique(), X_tr.shape[0]

(3941, 3941)

In [195]:
reset_seeds()
X_tr.drop('CustomerID',axis=1,inplace=True)

In [196]:
X_tr.head()

,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
1186,NaN,Computer,1,20.0,Credit Card,Female,2.0,3,Mobile Phone,3,Single,2,0,16.0,1.0,1.0,0.0,115
3145,7.0,Mobile Phone,1,7.0,Credit Card,Female,4.0,3,Grocery,3,Married,6,0,20.0,NaN,2.0,6.0,287
608,0.0,Mobile Phone,1,8.0,Credit Card,Male,2.0,1,Laptop & Accessory,2,Divorced,2,0,12.0,1.0,1.0,3.0,155
5202,18.0,Computer,1,12.0,Debit Card,Male,3.0,5,Mobile Phone,3,Single,5,0,15.0,1.0,2.0,4.0,157
4133,16.0,Mobile Phone,1,9.0,Debit Card,Male,3.0,4,Others,3,Married,7,0,NaN,2.0,5.0,15.0,314


In [197]:
reset_seeds()
X_te.set_index(['CustomerID'], inplace=True)
print(f'{X_te.shape}')
X_te.head()

(1689, 18)


,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
CustomerID,,,,,,,,,,,,,,,,,,
50175,24.0,Mobile Phone,1,8.0,Credit Card,Male,2.0,5,Laptop & Accessory,4,Married,6,0,18.0,0.0,1.0,2.0,155
55499,1.0,Phone,3,16.0,Credit Card,Male,3.0,4,Mobile Phone,1,Single,3,0,14.0,1.0,2.0,1.0,136
54706,4.0,Mobile Phone,3,34.0,E wallet,Male,4.0,4,Laptop & Accessory,1,Married,9,0,24.0,3.0,8.0,8.0,178
51155,7.0,Computer,1,NaN,CC,Male,2.0,3,Mobile,4,Married,2,1,12.0,1.0,2.0,3.0,123
54851,5.0,Mobile Phone,1,19.0,Credit Card,Female,3.0,4,Laptop & Accessory,1,Married,2,1,12.0,7.0,9.0,6.0,184


In [198]:
# 결측치, 이상치가 조정/제거된 전처리 데이터임

In [199]:
# 피처 생성

In [200]:
# 스케일링->SMOTE->모델링->CV->모델평가->모델 저장 내일 진행